# Optimizer bake-off on the kneed walker (Colab)

Compares five search methods at an equal **episode** budget on the 520-weight kneed brain:
`ga`, `cmaes`, `openai_es`, `es_raw` (the literal `theta += alpha/(n*sigma) * sum(F_i * eps_i)` update), and `random_search`.

**Runtime: CPU, not GPU.** The cost is Pymunk physics on the CPU; the 520-weight matmul is noise. A GPU runtime is still the right choice because it comes with more vCPUs.

Order of business: pilot (5 seeds, 5k episodes, ~10 min) → read the curves → full run (~4-6 h, resumable if Colab disconnects).

## 1. Get the code

In [ ]:
import os, subprocess

REPO   = "https://github.com/000wahab000/Evolution.git"
BRANCH = "feat/knees-v2"
DIR    = "/content/Evolution-1"

if not os.path.isdir(os.path.join(DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "pull", "--ff-only"], check=False)

os.chdir(DIR)
print("branch:", subprocess.run(["git", "branch", "--show-current"],
                                capture_output=True, text=True).stdout.strip())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 2. Preflight (cheap — catches a stale clone)

In [ ]:
import os, sys
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.research.optimizers import OPTIMIZERS

need    = ["experiments/pilot5_optimizers_knees.json", "experiments/exp5_optimizers_knees.json"]
missing = [p for p in need if not os.path.exists(p)]

if missing:
    raise RuntimeError(
        "Missing: " + ", ".join(missing) + ".\n"
        "Re-run cell 1 to pull the latest branch."
    )
if "es_raw" not in OPTIMIZERS:
    raise RuntimeError(
        "optimizers.py has no es_raw; it has " + str(sorted(OPTIMIZERS)) + ".\n"
        "Re-run cell 1 to pull the latest branch."
    )

print("preflight ok | repo:", os.getcwd(), "| optimizers:", sorted(OPTIMIZERS))

## 3. Install (research extras: cma, scipy, matplotlib)

In [ ]:
!pip -q install -e ".[research]"
!python -c "import numpy, pymunk, cma, scipy; print('numpy', numpy.__version__, '| pymunk', pymunk.version, '| cma', cma.__version__, '| scipy', scipy.__version__)"

## 4. Settings

Reports vCPUs and where results land. Mount Drive so a disconnect doesn't lose completed runs.

In [ ]:
import os, shutil, subprocess

def vcpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:
        return os.cpu_count() or 1

smi = shutil.which("nvidia-smi")
gpu = subprocess.run([smi, "-L"], capture_output=True, text=True).stdout.strip() if smi else ""

print("GPU    :", gpu or "none (fine — this workload is CPU-bound)")
print("vCPUs  :", vcpus(), "| repo:", os.getcwd())

MOUNT_DRIVE = True   # set False to keep results only in /content (lost on disconnect)
OUT = "/content/results"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/evolution_results"

os.makedirs(OUT, exist_ok=True)
print("Results:", OUT)

WORKERS = vcpus()
print("Workers:", WORKERS)

## 5. Pilot run (5 seeds × 5,000 episodes per variant)

25 runs, roughly 10 minutes on 8 vCPUs. Each run is skipped on re-run once it has a `DONE` file.

In [ ]:
!python scripts/run_experiment.py experiments/pilot5_optimizers_knees.json --workers {WORKERS} --out {OUT}

## 6. Analyze the pilot & price the full run

In [ ]:
!python scripts/analyze.py {OUT}/pilot5_optimizers_knees --no-replay

In [ ]:
import glob, json

def core_hours(exp):
    files = glob.glob(f"{OUT}/{exp}/*/seed_*/meta.json")
    return sum(json.load(open(f))["wall_s"] for f in files) / 3600

pilot_h = core_hours("pilot5_optimizers_knees")
print(f"pilot: {pilot_h:.2f} core-hours")
print(f"full exp5 = 6× seeds → {pilot_h*6:.1f} core-hours → ~{pilot_h*6/WORKERS:.1f} h on {WORKERS} workers")

### Gate: read this before spending the full budget

Open `pilot5_optimizers_knees/summary/REPORT.md` and `summary/figures/F1.png`.

- Every variant must show a rising `best_train` curve. A flat line means wrong plumbing or a wrong step size, not a losing method.
- `es_raw` must not diverge. If it does, lower `alpha` in both configs before the full run.
- Five seeds cannot support H11. The pilot is a cost estimate and a sanity check.

## 7. Full run (30 seeds × 25,000 episodes per variant)

150 runs. Re-run this cell after a disconnect — finished runs are skipped. Do not edit configs between pilot and full run.

In [ ]:
!python scripts/run_experiment.py experiments/exp5_optimizers_knees.json --workers {WORKERS} --out {OUT}

## 8. Analyze and read the verdict

F7/F9 re-simulate the saved champions. `H11` is the pre-registered contrast (openai_es vs es_raw); every other pair is exploratory.

In [ ]:
!python scripts/verify_results.py {OUT}/exp5_optimizers_knees
!python scripts/analyze.py {OUT}/exp5_optimizers_knees

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown("## Summary (IQM + 95% bootstrap CI)"))
s = pd.read_csv(f"{OUT}/exp5_optimizers_knees/summary/summary.csv")
cols = [c for c in ["variant", "n", "test_distance_iqm", "test_distance_ci_lo",
                    "test_distance_ci_hi", "fall_rate", "exploit_flag"] if c in s]
display(s[cols].round(2))

display(Markdown("## Pairwise contrasts"))
p = pd.read_csv(f"{OUT}/exp5_optimizers_knees/summary/pairwise.csv")
cols = [c for c in ["id", "a", "b", "metric", "verdict", "p_holm", "cliffs_delta"] if c in p]
display(p[cols])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/exp5_optimizers_knees/summary/REPORT.md").read()))

## 9. Take the results with you

In [ ]:
import os, shutil
from google.colab import files

zip_path = shutil.make_archive("exp5_optimizers_knees", "zip", f"{OUT}/exp5_optimizers_knees")
files.download(zip_path)